# 04 · Avaliação do pipeline RAG

Roda o conjunto `evaluation/questions.yaml` e analisa:
- decisões do guardrail (responder × recusar)
- precisão das evidências (proxy por regex de tema)
- extração e respeito aos filtros de metadados

> Para gerar respostas com o LLM, implemente `src/voc_rag/generation/llm.py` e use `generate=True`.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.max_colwidth", 200)

In [ ]:
import yaml
from voc_rag.pipeline import VoCPipeline

pipeline = VoCPipeline.from_settings()
questions = yaml.safe_load((ROOT / "evaluation/questions.yaml").read_text(encoding="utf-8"))["questions"]

In [ ]:
rows = []
for q in questions:
    r = pipeline.ask(q["question"], generate=False)
    sig = r.trace.guardrail["signals"]
    rows.append({"id": q["id"], "esperado": q["expected"], "status": r.status.value,
                 "margem_escopo": sig["scope_margin"], "suporte_lexical": sig["lexical_support"],
                 "relevantes": sig["relevant_count"], "motivo": r.trace.guardrail["reason"],
                 "filtros": r.trace.filters_applied})
res = pd.DataFrame(rows)
res

In [ ]:
ax = res.plot.scatter(x="margem_escopo", y="relevantes",
                      c=res["esperado"].map({"answer": "tab:blue", "refuse": "tab:red"}),
                      figsize=(7, 4), title="Responder (azul) × recusar (vermelho)")
ax.axvline(pipeline.settings.scope_margin, ls="--", c="gray")
res["acertou"] = (res["status"].isin(["out_of_scope", "insufficient_evidence"])) == (res["esperado"] == "refuse")
print("decisões corretas:", res["acertou"].sum(), "/", len(res))

## Inspeção de uma resposta completa

In [ ]:
r = pipeline.ask("Quais são os principais problemas relatados nas avaliações com nota baixa?")
print(r.status, "|", r.trace.filters_applied)
print(r.trace.context_preview[:3000])